# DuckDB + SLayer, from the command line

The same demo as the [Python notebook](duckdb_python_nb.ipynb), but set up and queried entirely through the `slayer` command line — no Python DuckDB calls, no long-running server. We register a DuckDB datasource, define a one-model semantic layer whose SQL reads the CSV **straight off the CDN**, then query it with `slayer query` (JSON in, JSON out).

**Prerequisites:** `pip install motley-slayer`.

## 1. Set it up with the CLI

Two `slayer` commands, no `duckdb` in sight. The datasource is an empty in-memory DuckDB; the model's SQL uses DuckDB's `read_csv_auto` to read the remote file live, so every query reaches over the wire. Columns are declared inline — the semantic layer in one small YAML.

In [1]:
%%bash
set -e
export SLAYER_STORAGE=.cache/cli/store
rm -rf .cache/cli && mkdir -p .cache/cli

# A DuckDB datasource — no database file, no `duckdb` call: the model's SQL
# reads the CSV straight off the CDN over httpfs, fresh on every query.
slayer datasources create "duckdb:///:memory:" --name weather_db -y

cat > .cache/cli/weather.yaml <<'YAML'
name: weather
data_source: weather_db
sql: SELECT * FROM read_csv_auto('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv')
columns:
  - {name: date, type: date}
  - {name: precipitation, type: number}
  - {name: temp_max, type: number}
  - {name: temp_min, type: number}
  - {name: wind, type: number}
  - {name: weather, type: string}
YAML

slayer models create .cache/cli/weather.yaml
slayer models list

Created datasource 'weather_db' (duckdb).


Created model 'weather'.


weather


## 2. A warm-up query

`slayer query` takes a JSON query and runs it. Average high and day count per weather type — `--format table` prints the CLI's own rendering.

In [2]:
%%bash
export SLAYER_STORAGE=.cache/cli/store
slayer query '{"source_model": "weather", "dimensions": ["weather"], "measures": [{"formula": "temp_max:avg", "name": "avg_high"}, {"formula": "*:count", "name": "days"}], "order": [{"column": "days", "direction": "desc"}]}' --format table

weather.weather | weather.avg_high | weather.days
--------------- | ---------------- | ------------


rain | 13.454602184087364 | 641
sun | 19.861875000000005 | 640
fog | 16.75742574257425 | 101
drizzle

 | 15.926415094339617 | 53
snow | 5.573076923076924 | 26

5 row(s)


## 3. The hero query — JSON in, JSON out, into pandas

A tiny wrapper runs `slayer query --format json` and parses the rows. The query is the same two-stage one as the Python notebook: stage 1 totals monthly rainfall and its value twelve months back (`time_shift(..., -1, 'year')`); stage 2 bands each month *rainy* / *dry* by that total inside a `CASE WHEN` dimension and regroups. The first year's year-over-year values are null — nothing precedes 2012.

In [3]:
import json
import os
import subprocess

import pandas as pd

STORAGE = ".cache/cli/store"


def slayer_query(query, *, dry_run=False):
    """Run `slayer query` with a JSON query; return the rows (or the SQL)."""
    cmd = ["slayer", "query", json.dumps(query)]
    cmd += ["--dry-run"] if dry_run else ["--format", "json"]
    result = subprocess.run(
        cmd, capture_output=True, text=True, env={**os.environ, "SLAYER_STORAGE": STORAGE}
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout if dry_run else json.loads(result.stdout)

In [4]:
hero = [
    {
        "name": "monthly",
        "source_model": "weather",
        "time_dimensions": [{"dimension": "date", "granularity": "month"}],
        "measures": [
            {"formula": "precipitation:sum", "name": "rain"},
            {
                "formula": "precipitation:sum - time_shift(precipitation:sum, -1, 'year')",
                "name": "rain_yoy",
            },
        ],
    },
    {
        "source_model": "monthly",
        "dimensions": [
            {
                "expression": "CASE WHEN rain > 100 THEN 'rainy' ELSE 'dry' END",
                "name": "month_type",
            },
            "date",
        ],
        "measures": [
            {"formula": "rain:sum", "name": "total_rain"},
            {"formula": "rain_yoy:sum", "name": "total_rain_yoy"},
        ],
        "order": [{"column": "date", "direction": "asc"}],
    },
]

df = pd.DataFrame(slayer_query(hero))

n_null_yoy = int(df["monthly.total_rain_yoy"].isna().sum())
assert len(df) == 48, f"expected 48 monthly rows, got {len(df)}"
assert n_null_yoy == 12, f"expected 12 null YoY rows (first year), got {n_null_yoy}"
df

,monthly.month_type,monthly.date,monthly.total_rain,monthly.total_rain_yoy
0,rainy,2012-01-01 00:00:00,173.3,NaN
1,dry,2012-02-01 00:00:00,92.3,NaN
2,rainy,2012-03-01 00:00:00,183.0,NaN
3,dry,2012-04-01 00:00:00,68.1,NaN
4,dry,2012-05-01 00:00:00,52.2,NaN
5,dry,2012-06-01 00:00:00,75.1,NaN
6,dry,2012-07-01 00:00:00,26.3,NaN
7,dry,2012-08-01 00:00:00,0.0,NaN
8,dry,2012-09-01 00:00:00,0.9,NaN
9,rainy,2012-10-01 00:00:00,170.3,NaN


## 4. The SQL that ran

`--dry-run` returns the single SQL statement SLayer generated for the whole two-stage query without executing it.

In [5]:
sql = slayer_query(hero, dry_run=True)
assert "SELECT" in sql, "expected SQL from the dry run"
print(sql)

WITH monthly AS (
  SELECT
    _stage_inner."weather.date" AS "date",
    _stage_inner."weather.rain" AS "rain",
    _stage_inner."weather.rain_yoy" AS "rain_yoy"
  FROM (
    SELECT
      "weather.date",
      "weather.rain",
      "weather.rain_yoy"
    FROM (
      WITH base AS (
        SELECT
          DATE_TRUNC('MONTH', weather.date) AS "weather.date",
          CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.rain"
        FROM (
          SELECT
            *
          FROM READ_CSV_AUTO('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv')
        ) AS weather
        GROUP BY
          DATE_TRUNC('MONTH', weather.date)
      ), shifted__time_shift_inner AS (
        SELECT
          DATE_TRUNC('MONTH', weather.date) + INTERVAL '1' YEAR AS "weather.date",
          CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.rain"
        FROM (
          SELECT
            *
          FROM READ_CSV_AUTO('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/se

---

The whole semantic layer over a file on the internet — three shell commands and a JSON query. See the [Python notebook](duckdb_python_nb.ipynb) for the in-process library version with live schema auto-ingestion, and [formulas](../../concepts/formulas.md) for the full transform vocabulary.